In [27]:
import pandas as pd

In [28]:
# import our saved csv files from last week
listing = pd.read_csv('idx_files/w2_residential_listing.csv')
sold = pd.read_csv('idx_files/w2_residential_sold.csv')

/var/folders/15/fxy7kh_n2sbbwk9d_48fvvh40000gn/T/ipykernel_82273/1171676499.py:2: DtypeWarning: Columns (2,39) have mixed types. Specify dtype option on import or set low_memory=False.
  listing = pd.read_csv('idx_files/w2_residential_listing.csv')
/var/folders/15/fxy7kh_n2sbbwk9d_48fvvh40000gn/T/ipykernel_82273/1171676499.py:3: DtypeWarning: Columns (0,1,7,51,63,64,65) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('idx_files/w2_residential_sold.csv')


In [29]:
# fetch the mortgage csv from FRED
df_mortgage = pd.read_csv('idx_files/MORTGAGE30US.csv', parse_dates=['observation_date'])
df_mortgage.columns = ['date', 'rate_30yr_fixed']

In [49]:
df_mortgage

,date,rate_30yr_fixed,year_month
0,1971-04-02,7.33,1971-04
1,1971-04-09,7.31,1971-04
2,1971-04-16,7.31,1971-04
3,1971-04-23,7.31,1971-04
4,1971-04-30,7.29,1971-04
...,...,...,...
2880,2026-06-11,6.52,2026-06
2881,2026-06-18,6.47,2026-06
2882,2026-06-25,6.49,2026-06
2883,2026-07-02,6.43,2026-07


In [30]:
# resampling weekly rates to monthly averages
df_mortgage['year_month'] = df_mortgage['date'].dt.to_period('M')
monthly = df_mortgage.groupby('year_month')['rate_30yr_fixed'].agg('mean').reset_index()

In [48]:
monthly

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975
...,...,...
659,2026-03,6.1775
660,2026-04,6.3320
661,2026-05,6.4425
662,2026-06,6.4900


In [31]:
# key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

In [32]:
# key off ListingContractDate
listing['year_month'] = pd.to_datetime(listing['ListingContractDate']).dt.to_period('M')

In [33]:
# merge listing and sold with mortgages
sold_rates = sold.merge(monthly, on='year_month', how='left')
listing_rates = listing.merge(monthly, on='year_month', how='left')

In [45]:
listing_rates

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.00,177861.0,2220 Avenue Of The Stars 2704,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,16 Palisades,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,1615 Waverly Road,2024-01,6.6425
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,False,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,2250 Indian Creek Road,2024-01,6.6425
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,False,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,317 E. Bayfront,2024-01,6.6425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615683,849000.0,1156787989,NaN,NaN,NaN,Edward,Zamora,37.950161,-121.659890,1691 Eden Plains Rd,...,False,2.0,NaN,94513,NaN,NaN,20909.0,1691 Eden Plains Rd,2026-06,6.4900
615684,1375000.0,1156489602,NaN,2026-06-26,1405000.0,Jezer,Manalastas,32.629353,-116.949771,1561 Copper Penny Dr,...,False,3.0,NaN,91915,Keller Williams San Diego Metro,125.00,NaN,1561 Copper Penny Dr,2026-06,6.4900
615685,649900.0,1154722990,NaN,NaN,NaN,Benjamin,Diaz-Habben,38.114716,-122.239396,21 Monte Vista Ave,...,False,0.0,NaN,94590,NaN,NaN,7405.0,21 Monte Vista Ave,2026-06,6.4900
615686,795000.0,1153448926,NaN,NaN,NaN,Jace,Coleman,32.837643,-116.765217,1242 Marshall Rd,...,False,1.0,NaN,91901,NaN,NaN,37897.0,1242 Marshall Rd,2026-06,6.4900


In [36]:
sold_rates.rate_30yr_fixed.isna().sum()

np.int64(0)

In [38]:
listing_rates.rate_30yr_fixed.isna().sum()

np.int64(0)

In [41]:
print(sold_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head(16))

     CloseDate year_month  ClosePrice  rate_30yr_fixed
0   2024-01-26    2024-01    240000.0           6.6425
1   2024-01-05    2024-01    815000.0           6.6425
2   2024-01-05    2024-01    810000.0           6.6425
3   2024-01-30    2024-01    858000.0           6.6425
4   2024-01-29    2024-01   1890500.0           6.6425
5   2024-01-02    2024-01   2100000.0           6.6425
6   2024-01-22    2024-01   1950000.0           6.6425
7   2024-01-31    2024-01   2340000.0           6.6425
8   2024-01-31    2024-01   1485000.0           6.6425
9   2024-01-30    2024-01   1130000.0           6.6425
10  2024-01-16    2024-01   1060000.0           6.6425
11  2024-01-31    2024-01   2150000.0           6.6425
12  2024-01-29    2024-01   2000000.0           6.6425
13  2024-01-24    2024-01   1939716.0           6.6425
14  2024-01-31    2024-01   1080000.0           6.6425
15  2024-01-31    2024-01    920000.0           6.6425


In [47]:
print(listing_rates[['ListingContractDate', 'year_month', 'ListPrice', 'rate_30yr_fixed']].head(16))

   ListingContractDate year_month   ListPrice  rate_30yr_fixed
0           2024-01-01    2024-01   1340000.0           6.6425
1           2024-01-24    2024-01   2500000.0           6.6425
2           2024-01-12    2024-01   3150000.0           6.6425
3           2024-01-20    2024-01   3090000.0           6.6425
4           2024-01-12    2024-01  12725000.0           6.6425
5           2024-01-30    2024-01    665000.0           6.6425
6           2024-01-26    2024-01   1125000.0           6.6425
7           2024-01-08    2024-01   1849000.0           6.6425
8           2024-01-25    2024-01   1389000.0           6.6425
9           2024-01-29    2024-01   1249888.0           6.6425
10          2024-01-09    2024-01    349000.0           6.6425
11          2024-01-01    2024-01    950000.0           6.6425
12          2024-01-27    2024-01   1595000.0           6.6425
13          2024-01-31    2024-01   1299000.0           6.6425
14          2024-01-22    2024-01    650000.0          